# DCM expert-layer update and DCM v2 survey design

*With one upstream tree-prior diagnostic*

**Date**: 27 April 2026  
**Format**: ~5 min read; ~20 min discussion

Mostly a calibration check, structured against your original notes:

1. Where we left off
2. **Expert-specific parameters** — how I implemented your original spec
3. **Reference systems** — how I went about your original spec
4. **Survey-design implications** for DCM v2
5. The three-state ordinal leaf — brief note
6. One upstream tree-prior diagnostic
7. What I want from this meeting

*Not asking for tree-prior model selection today — parking that pending Matilda's restructure (see §6).*

## 1. Where we left off

The midterm (Mar 31) covered the bottom-layer setup: ordered probit replacing the binary-collapse + outer-loop simulation, joint multi-system fit, initial PPC. The week-7 report (Apr 24) added the three-state leaf (§5) and showed posterior `C` across all 13 stances.

Today: calibrate on (a) whether the bottom-layer direction matches what you originally had in mind, (b) survey-design recommendations for DCM v2, (c) integration with Matilda / Edwin / Andrew per the week-7 next-actions list. Default plan for remaining SPAR weeks: write up the expert-layer + survey-design content and coordinate with the team — unless you redirect.

Numbers below are the baseline GWT fit from the midterm (anchored Human/ELIZA, Chicken/LLMs free).

## 2. Expert-specific parameters: how I implemented your original spec

**Headline**: per-expert parameters are computationally feasible and produce real posterior movement, but under current rater × system coverage they are high-leverage enough that I'd treat them as sensitivities, not defaults.

**Your original note**: latent signal `s = δ_k^(Z) + ε`, with `δ_k^(1)` and `δ_k^(0)` as expert k's average signal when the indicator is truly present vs truly absent. The difference `δ_k^(1) − δ_k^(0)` is the per-rater discriminability. Cutpoints shared, or expert-specific depending on data availability.

What's in the model now and what I tested:

| Parameter | Status | What happens |
|---|---|---|
| Per-expert location shift `b_e` (per-expert intercept; shared slope) | Implemented; available as a flag | Captures an expert's baseline response tendency on the ordinal scale, but NOT per-rater discriminability. Even with a narrow prior (σ=0.5), the model attributes a large positive shift (~+1.0) to the rater who only scores Chicken — and Chicken `C` drops from 0.30 (baseline) to 0.17. So `b_e` is doing real work on a headline number. |
| Per-expert noise scale `σ_e` | Tested as a sensitivity | Computationally viable. Posterior concentrates around plausibly-different rater scales (highest ~3.2; lowest ~0.4). Moves Chicken `C` by **−0.08** (beyond the ±0.05 range I'd want as a default). Variance over-prediction at reference cells worsens about 3×. |
| Per-expert ordered cutpoints (hierarchical) | Tested as a sensitivity | Closer to your "expert-specific cutpoints depending on data availability". Improves the worst reference-cell predictions modestly; trades off elsewhere; pushes Chicken `C` to the ±0.05 edge. |

Across all three: Chicken `C` moves downward by ~0.05–0.13, with the largest movement under the `b_e` fit. This is not a failure of the expert-specific model — the posteriors concentrate fine. It's a design-level point: the model's attribution of variation to rater effects vs. system `C` is high-leverage under current rater × system coverage. A rater who only scores Chicken can't cleanly identify "rater is lenient" vs "Chicken genuinely scores high" without more cross-system overlap. The pattern is evidence that richer expert calibration needs more overlap (§4), not that the parameters fail.

**The full two-mean parameterisation you originally specified — separate per-rater `δ_k^(0)` and `δ_k^(1)` (so per-rater discriminability `δ_k^(1) − δ_k^(0)` can vary) — is not in the current model.** Identifying per-rater discriminability needs each rater scoring multiple indicators across known-state systems; currently most raters don't have that breadth.

## 3. Reference systems: how I went about your original spec

**The question I'd most value your view on**: the joint-tree-fit setup — single tree per stance shared across all four systems, with only `C` differing per system. This hasn't been an explicit design discussion. Comfortable with this as the default? It's doing what your reference-systems spec asked for in spirit (reference data updates lower-layer `β` parameters), but worth confirming the multi-system structure matches your intent.

**Your original note**: graded credences across multiple reference systems, with the kickoff bracket of "can bracket any systems that are not human + thermostat" for the initial fit. Reference data updates the base-rate posterior AND lower-layer parameters. Diffuse hyperprior to avoid overfitting to a small reference class.

Per your bracket, I implemented the Human/ELIZA-only version: hard anchors at `C=0.999` and `C=0.001`. Joint multi-system fit so reference data DOES update lower-layer `β`. The mid-range systems (dogs, octopuses) become a survey-design recommendation in §4.

Sensitivity: refit with soft anchors `Beta(50, 1)` on Human, `Beta(1, 50)` on ELIZA. Posteriors land near the hard values (Human ~0.99, ELIZA ~0.01); Chicken / LLMs essentially unchanged. The audit doesn't suggest the hard anchors are driving the downstream results — though Beta(50,1) and Beta(1,50) are themselves quite informative priors, so this is reassurance rather than a strong test.

**One known divergence from your original note**: you mentioned a Beta hyperprior on the base rate `π₀`. The current model uses `Beta(1, 5)` per system without a shared `π₀` hyperprior across systems. Worth flagging — small change, but I haven't done it. Easy to add if you'd like it.

## 4. Survey-design implications for DCM v2

Probably the highest-value thing this round of modelling produced. The richer per-expert parameters (§2) and a richer reference-system anchor set (§3) are both blocked by the same data limitation: rater × system coverage is too thin to identify the parameters cleanly. Concrete recommendations, each tied back to a specific finding:

| Current limitation | What it does to the model | What I'd recommend for future data |
|---|---|---|
| 1 rater on each of Human / ELIZA | Per-expert parameters get high leverage from a single rater; can't separate "this rater is noisier" from "this system just differs". | ≥2-3 raters per reference system. |
| Thin rater × system overlap | Rater effects trade off with system `C`; richer per-expert models become unstable at the system level. | Each rater scores multiple systems, including ≥1 reference system. |
| Only 2 reference systems, both extreme anchors | No mid-range data to identify rater discriminability away from saturation. | Include at least one mid-range reference system (dogs, octopuses, simpler invertebrates). Closer to your original spec. |
| Forced ordinal responses | Low-confidence ratings indistinguishable from middle ratings. | Optional "not enough information" / confidence option alongside the 7-point scale. |
| Per-rater discriminability (`δ_k^(1) − δ_k^(0)`) not in current model | The full spec needs per-rater breadth to identify slope. | The reference-system overlap above is the prerequisite. |

With the right data structure, the next round of modelling can be substantially richer than what's possible now.

## 5. Three-state ordinal leaf — brief note

One small leaf-model change since the midterm: the per-indicator latent has 3 states (`m_j ∈ {0, 1, 2}` via `Binomial(2, q_j)`) instead of 2.

Why: the cross-system rater puts 86% mass on rating 7 at Human and 96% mass on rating 1 at ELIZA. A 2-state latent can't reproduce that much extreme-tail mass at any cutpoint configuration; 3 states can. Treat it as a flexible mixture, not a literal claim that indicators are 3-valued. Sampling clean.

## 6. One upstream tree-prior diagnostic

While diagnosing leaf-side misfit at the cross-system rater's reference cells, I noticed an upstream issue worth flagging.

**The diagnostic**: before any rating data, the current GWT tree prior moves only ~5% of the Human-vs-ELIZA root gap to the indicator layer. (Mean `q_j(C=0.999) − q_j(C=0.001) ≈ 0.048` over 10⁴ Monte Carlo draws from the prior.) So the prior puts most of the reference-system anchoring burden on the rating layer downstream — which interacts with the bottom-layer fit issues in §2.

**What I tried**: a label-level pooling sensitivity — same `(support, demandingness)` labels you specified, but letting same-label edges share a calibrated edge weight rather than fixed paper means. It improves some predictive checks; the first version had an absence-sharing artefact (one cluster's apparent shift was inherited from a shared parameter, not direct evidence); a cleaner version is more conservative on `C` but loses some predictive gain.

**I do not want to present this as a settled replacement model.** Worth discussing whether this should fold into Matilda's tree restructure (which is much more substantive than my mechanical pooling tweak) rather than be pursued as a standalone branch. Backup notes available if you want to deep-dive.

## 7. What I want from this meeting

1. **Calibration on the bottom-layer direction.** Are you happy with ordinal + three-state + per-expert location shifts as the bottom-layer direction? Anything in §2 or §3 you'd want changed?

2. **Survey-design alignment.** Does the table in §4 match what RP is realistically planning for DCM v2? Anything that doesn't fit the data-collection plans on your side?

3. **Remaining SPAR time.** Should it prioritise (a) writing up the expert-layer + survey-design content, (b) tree-prior diagnostics, or (c) coordination with Matilda / Edwin? My own lean is (a) + (c) given §6.

4. **Tree-prior diagnostic into Matilda's restructure?** Should the §6 finding feed into her work rather than be a standalone branch?

5. **Edwin's JAX backend status?** It's the prerequisite for proper partial pooling on the tree side and for the richer per-expert work in §2. Worth coordinating now or wait?

6. **(Time permitting)** Is there a sensible way for me to keep contributing to the DCM after SPAR? Happy to discuss verbally — I'd also value a quick conversation on AI welfare / digital minds as a longer-term direction, given how few entry points there are. Not for the written record.

---

## Appendix — backup numbers

Available if you want to deep-dive. Not part of the main flow. Rater anonymisation: A through F mapped consistently across all tables below (A is the same person in every table; A is the cross-system rater).

**Per-expert noise scales `σ_e`** (sensitivity fit, hierarchical partial pooling, three-state leaf, paper tree, joint anchored):

| Rater | σ_e median | 94% interval |
|---|---:|---|
| A | 3.24 | [2.53, 4.25] |
| B | 1.40 | [1.02, 1.96] |
| C | 1.19 | [0.88, 1.59] |
| D | 0.70 | [0.41, 1.03] |
| E | 0.65 | [0.46, 0.89] |
| F | 0.44 | [0.21, 0.70] |

System-`C` shift vs no-σ_e three-state baseline: Chicken 0.297 → 0.217 (Δ = −0.08), LLMs 0.118 → 0.102 (Δ = −0.016). Sampling: 1 divergence, max R-hat 1.000, min ESS bulk 4135.

**Hierarchical per-expert cutpoints** (sensitivity fit): improves Rater A's reference-cell predicted means (e.g., Rater A × Human tree-implied pred 3.94 → 4.47 vs observed 5.78). System `C` shift vs the *binary* baseline (this fit was binary-leaf; numbers therefore on a different baseline): Chicken 0.249 → 0.199 (at the ±0.05 guardrail edge); global mean PPC degrades elsewhere. Sampling clean. Worth re-running on the three-state baseline if we want apples-to-apples with the other sensitivities.

**Per-expert location shift `b_e` demonstration fit** (`EXPERT_SHIFT_SIGMA=0.5` narrow prior, three-state leaf, paper tree, joint anchored, 4 chains × 2000 draws):

| Rater | b_e median | 94% interval | Note |
|---|---:|---|---|
| A | 0 (anchor) | — | reference rater (cross-system) |
| B | +0.03 | [-0.35, +0.42] | scores LLMs only; shift consistent with zero |
| C | +0.18 | [-0.17, +0.54] | scores LLMs only; modest positive |
| D | **+1.03** | [+0.60, +1.46] | scores Chicken only; large positive — drives the headline `C` shift |
| E | +0.28 | [-0.08, +0.63] | scores LLMs only; modest positive |
| F | +0.48 | [+0.03, +0.91] | scores Chicken only; modest positive |

Headline `C` posteriors (vs no-`b_e` three-state baseline):
- Human: 0.999 (anchored, unchanged)
- Chicken: **0.168** [0.009, 0.535] vs baseline 0.297 — **Δ = −0.13**
- LLMs: 0.122 [0.006, 0.447] vs baseline 0.118 — Δ = +0.004
- ELIZA: 0.001 (anchored, unchanged)

Sampling: 0 divergences, max R-hat 1.000, min ESS bulk 10349.

The Chicken shift comes from Rater D's large positive `b_e`: D scores Chicken indicators with a systematically high baseline, and once the model attributes some of that to the rater's response tendency rather than the underlying indicator state, the inferred `C_chicken` drops. Rater F (also Chicken-only) has a more modest shift. Whether D's high baseline reflects a real rater-level tendency vs. genuine Chicken signal can't be disentangled without more cross-system overlap from D.

**Reference-systems soft-anchor audit**: Beta(50, 1) on Human, Beta(1, 50) on ELIZA. Human posterior median 0.990 [0.951, 1.000]; ELIZA 0.010 [0.000, 0.049]. Chicken / LLMs essentially unchanged from the hard-anchor fit.

**Tree-prior diagnostic + pooling sensitivity**: detailed numbers and the latent-state-tree alternative comparison live in `meeting_note_arvo_2026-04-27.md` and `diagnostics_round2_results.md`.

## Appendix — tree-prior deep-dive (if we go there)

The compact backup if we want to discuss the tree work in detail.

### A. The diagnosis in numbers

10⁴ Monte Carlo draws from the paper prior (no ratings observed), propagating `q_j` through the GWT tree at three values of root `C`:

| Root C | mean indicator `q_j` | min | max |
|---|---:|---:|---:|
| 0.001 (ELIZA-like) | 0.595 | 0.286 | 0.842 |
| 0.500 | 0.619 | 0.298 | 0.860 |
| 0.999 (Human-like) | 0.643 | 0.288 | 0.880 |

Mean indicator-layer separation between Human and ELIZA anchors: **0.048** on a root anchor gap of 0.998. Depth-2 indicators get ~13% transmission; depth-3 get ~5%. The compounding `(β_pres − β_abs)` along the path explains it.

A separate check: posterior `δ_j` (path-product slope) under the joint anchored fit tracks the prior `δ_j` with ~+20% mean amplification (median log ratio +0.20). So the small posterior `δ_j` isn't sparse-data shrinkage — it's the prior structure compounded over depth.

### B. The pooling intervention

Each `(support, demandingness)` group gets ONE `β_pres` random variable, shared across all tree edges with that label. Logit-Normal prior, σ=0.5, median-centred at the paper Beta's prior mean. Same structure for `β_abs` (with the paper's d-only grouping). Three-state ordinal leaf as before.

Conceptually: keep the labels intact but let same-label edges *learn* their shared edge weight from data rather than having it hand-set.

### C. The weak-undermining cluster — the concrete example of what went wrong with d-only β_abs sharing

5 indicators sit under the subfeature "Autonomous Subparts" with labels `weak undermining + neutral`. Under the paper prior these have a slightly negative path-product slope `δ_j`. Cluster-mean `δ_j` summary:

| Fit | cluster mean δ̄ [94% CI] | Pr(δ̄ > 0) |
|---|---:|---:|
| paper prior | ~ -0.07 (prior, by construction) | low |
| `pool_3s` (label pooling, d-only β_abs) | +0.060 [-0.025, +0.165] | **0.90** |
| `pool_3s_abs_by_sd` ((s,d)-keyed β_abs) | +0.006 [-0.067, +0.088] | 0.56 |

Decomposition (counterfactual: hold pool fit posterior draws fixed but reset `β_abs__neutral` to its paper value 0.5 per draw, recompute cluster `δ_j`): **70% of the apparent positive shift under `pool_3s` is inherited from the shared `β_abs__neutral` posterior moving** — driven by data from the OTHER 10 neutral-demand cells (mostly strong/moderate-support paths). The weak-undermining cluster's own `β_pres` barely moves.

Once `β_abs` is keyed by `(s, d)` rather than `d` alone, that inheritance route closes, and the cluster collapses to sign-ambiguous. So `pool_3s_abs_by_sd` is the cleaner version of the pooling intervention — at the cost of losing most of the focus-cell PPC gain (see D).

### D. Headline numbers across fits

| Fit | What's different | C_Chicken | C_LLMs | E_cross × Human Δright |
|---|---|---:|---:|---:|
| `baseline_3s` | three-state leaf, paper tree | 0.297 | 0.118 | -0.228 |
| `pool_3s` | + label-pooling, d-only β_abs | **0.362** | **0.170** | **-0.163** (28% closure) |
| `pool_3s_abs_by_sd` | + (s,d)-keyed β_abs (cleaner) | 0.300 | 0.127 | -0.206 (9%) |
| latent-state-tree alt. | exact tree marginalisation, + (s,d) β_abs | **0.252** | **0.111** | -0.283 (-36% closure) |

The pooling moves headline `C` UP from baseline, partly through the inheritance artefact in C; the (s,d)-keyed version pulls it back; the latent-state-tree alternative pulls it down further still (more on this below).

### E. The two interpretations of internal nodes

While implementing the tree work I noticed the current code's tree structure is consistent with two different generative interpretations:

1. **Propagation device** (current implementation): internal feature/subfeature nodes are bookkeeping for marginal indicator probabilities. There are no shared latent binary variables at the feature/subfeature level mediating sibling indicators. Indicators are conditionally independent given `(C, β)`.
2. **Latent-state tree** (paper text could read this way too): internal nodes are real binary `z_v` random variables. Sibling indicators are correlated through shared latent ancestors.

These are materially different DGPs (Δℓ ~10 nats library-wide across the composite fits I'd run vs. the exact-tree alternative). The latent-state version is more conservative on `C` because it accounts for evidence redundancy through shared latents. I don't have a strong epistemic preference for "Selective Attention is on/off" as a real binary state — propagation device feels closer to what we're actually doing — but it's a real choice the project should be explicit about.

### F. Caveats

- **Pooling fixes posterior label calibration, not prior-predictive mean transmission.** σ_pool sweep across {0.25, 0.5, 0.75, 1.0} shows mean prior-predictive `δ_j` barely moves (~0.04 throughout). What changes is the spread / sign-flip rate. Pooling is a calibration tool, not a prior-strength tool.
- **Focus-cell PPC trade-off is real.** Better PPC under `pool_3s` partly comes from the inheritance pattern that section C critiques. The "right" PPC for the latent-state-tree alternative would re-derive `ρ_m` accounting for joint tree structure (I implemented this; numbers similar to the composite-style PPC for E_cross × Human, slightly better for E_cross × ELIZA).
- **All-stance generalisation**: I haven't refit the 12 non-GWT stances under any of the new tree variants. The tree-pool branch is structure-agnostic so this would be a batch run, not new model code.

### G. Why I'm not pushing this today

Three reasons:
1. Pooling has the inheritance artefact (section C). The cleaner version loses most of the PPC gain. So neither variant is a strict improvement over baseline.
2. The deeper choice between the two DGPs (section E) is a project-level call, not a SPAR-fellow call.
3. Matilda's restructure is a much more substantive intervention on the tree side. My label-pooling tweak should probably feed into her work rather than be a parallel branch — see ask 4 in §7.